In [1]:
#is used in Python to define abstract base classes
#ABC
'''
Stands for Abstract Base Class.

You inherit from ABC to create a class that cannot be instantiated directly and can enforce method implementation in subclasses.

abstractmethod

A decorator used to mark a method as abstract.

Subclasses must implement this method; otherwise, they cannot be instantiated.
'''

'\nStands for Abstract Base Class.\n\nYou inherit from ABC to create a class that cannot be instantiated directly and can enforce method implementation in subclasses.\n\nabstractmethod\n\nA decorator used to mark a method as abstract.\n\nSubclasses must implement this method; otherwise, they cannot be instantiated.\n'

In [2]:
from abc import ABC, abstractmethod

In [3]:
from abc import ABC, abstractmethod

# Abstract base class representing a "runnable" component
# Any class that inherits from Runnable must implement the invoke() method
class Runnable(ABC):

    @abstractmethod
    def invoke(input_data):
        """
        Abstract method that defines the interface for running the component.
        
        Args:
            input_data: The input data that the component will process.
            
        Returns:
            The output of the component (any type depending on implementation).
        
        Notes:
            - This method must be implemented by all subclasses.
            - Provides a standard interface so different components can be used interchangeably.
        """
        pass

In [4]:
# Mock/fake LLM class implementing the Runnable interface
import random
class NakliLLM(Runnable):

    # Constructor: called when an instance is created
    def __init__(self):
        print('LLM created')

    # Mandatory method from Runnable ABC
    def invoke(self, prompt):
        """
        Process the input prompt and return a random response.
        Args:
            prompt (str): The input prompt/question
        Returns:
            dict: A dictionary containing a random response
        """
        response_list = [
            'Delhi is the capital of India',
            'IPL is a cricket league',
            'AI stands for Artificial Intelligence'
        ]
        # Randomly select a response from the list
        return {'response': random.choice(response_list)}

    # Optional method to mimic older predict-based interface
    def predict(self, prompt):
        """
        Legacy method for backward compatibility.
        Does the same thing as invoke.
        """
        response_list = [
            'Delhi is the capital of India',
            'IPL is a cricket league',
            'AI stands for Artificial Intelligence'
        ]
        return {'response': random.choice(response_list)}

In [5]:
class NakliPromptTemplate(Runnable):

  def __init__(self, template, input_variables):
    self.template = template
    self.input_variables = input_variables

  def invoke(self, input_dict):
    return self.template.format(**input_dict)

  def format(self, input_dict):
    return self.template.format(**input_dict)

In [6]:
class NakliStrOutputParser(Runnable):

  def __init__(self):
    pass

  def invoke(self, input_data):
    return input_data['response']

In [7]:
# RunnableConnector allows connecting multiple Runnable components sequentially
class RunnableConnector(Runnable):

    # Constructor: accepts a list of Runnable components
    def __init__(self, runnable_list):
        """
        Args:
            runnable_list (list): List of Runnable instances to be executed in sequence
        """
        self.runnable_list = runnable_list  # Store the components

    # Implements the abstract method from Runnable
    def invoke(self, input_data):
        """
        Sequentially runs all Runnables in the list.

        Args:
            input_data: Initial input to the first Runnable

        Returns:
            The output from the last Runnable in the sequence
        """

        # Iterate through each Runnable in order
        for runnable in self.runnable_list:
            # Pass the current input_data to the Runnable and update input_data with its output
            input_data = runnable.invoke(input_data)

        # Return the final output after all Runnables have processed the data
        return input_data

In [8]:
template = NakliPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length', 'topic']
)

In [9]:
llm = NakliLLM()

LLM created


In [10]:
parser = NakliStrOutputParser()

In [11]:
chain = RunnableConnector([template, llm, parser])

In [12]:
chain.invoke({'length':'long', 'topic':'india'})

'Delhi is the capital of India'

In [15]:
#We can combine multiple chains together
#Runnable helps in doing this

In [16]:
template1 = NakliPromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

In [17]:
template2 = NakliPromptTemplate(
    template='Explain the following joke {response}',
    input_variables=['response']
)

In [18]:
llm = NakliLLM()

LLM created


In [19]:
parser = NakliStrOutputParser()

In [20]:
chain1 = RunnableConnector([template1, llm])

In [21]:
chain2 = RunnableConnector([template2, llm, parser])

In [23]:
#Runnable which connect multiple runnables together
final_chain = RunnableConnector([chain1, chain2])

In [24]:
final_chain.invoke({'topic':'cricket'})

'AI stands for Artificial Intelligence'